In [14]:
from ccdc import io
from pathlib import Path
from ccdc.io import CrystalReader
from rdkit import Chem
from rdkit.Chem import Draw, AllChem
from IPython.display import display
from ccdc.io import MoleculeReader
import pandas as pd
import base64
from io import BytesIO
from IPython.display import display, HTML
from ccdc.io import CrystalReader
from rdkit.Chem import MolFromSmiles, MolToSmiles
from rdkit.Chem import Descriptors, rdMolDescriptors

In [6]:
csd = pd.read_csv('csd_hits.csv')
pdb = pd.read_csv('pdb_hits.csv')
all_db = pd.read_csv('all_hits.csv')
ttd = pd.read_csv('ttd_drug_hits.csv')

In [4]:
def largest_fragment_from_smiles(smi: str):
    """
    Returns (mol, clean_smiles) where mol is the RDKit molecule for the
    largest fragment (by heavy atoms) and clean_smiles is canonical SMILES.

    """
    if smi is None or (isinstance(smi, float) and pd.isna(smi)):
        return None, None

    smi = str(smi).strip()
    if not smi:
        return None, None

    mol = MolFromSmiles(smi)
    if mol is None:
        return None, None

    frags = Chem.GetMolFrags(mol, asMols=True, sanitizeFrags=False)
    if not frags:
        return None, None

    largest = max(frags, key=lambda m: m.GetNumHeavyAtoms())

    try:
        Chem.SanitizeMol(largest)
    except Exception:
        return None, None

    clean = MolToSmiles(largest, canonical=True, isomericSmiles=True)
    return largest, clean

In [5]:
def add_clean_smiles(df: pd.DataFrame, smiles_col: str = "SMILES", prefix: str = "") -> pd.DataFrame:
    out = df.copy()

    cols = out[smiles_col].apply(lambda s: pd.Series(largest_fragment_from_smiles(s)))
    cols.columns = [f"{prefix}mol", f"{prefix}clean_smiles"]

    out = pd.concat([out, cols], axis=1)
    out = out.dropna(subset=[f"{prefix}mol"]).reset_index(drop=True)

    print(f"[{prefix or 'df'}] Valid molecules: {len(out)} / {len(df)}")
    return out

In [19]:
csd_unique_clean = add_clean_smiles(csd, smiles_col="SMILES", prefix="")
all_db_clean   = add_clean_smiles(all_db,smiles_col="SMILES", prefix="")
ttd_clean = add_clean_smiles(ttd,smiles_col="canon_smiles", prefix="")

[14:39:26] WARNING: not removing hydrogen atom without neighbors
[14:39:26] WARNING: not removing hydrogen atom without neighbors
[14:39:26] WARNING: not removing hydrogen atom without neighbors
[14:39:26] WARNING: not removing hydrogen atom without neighbors
[14:39:26] WARNING: not removing hydrogen atom without neighbors
[14:39:26] WARNING: not removing hydrogen atom without neighbors
[14:39:26] WARNING: not removing hydrogen atom without neighbors
[14:39:26] WARNING: not removing hydrogen atom without neighbors
[14:39:26] WARNING: not removing hydrogen atom without neighbors
[14:39:26] WARNING: not removing hydrogen atom without neighbors
[14:39:26] WARNING: not removing hydrogen atom without neighbors
[14:39:26] WARNING: not removing hydrogen atom without neighbors
[14:39:26] WARNING: not removing hydrogen atom without neighbors
[14:39:26] WARNING: not removing hydrogen atom without neighbors
[14:39:26] WARNING: not removing hydrogen atom without neighbors
[14:39:26] WARNING: not r

[df] Valid molecules: 4971 / 4971
[df] Valid molecules: 238 / 238


KeyError: 'canon_smiles'

In [16]:
csd_unique_clean.head()

,CSD_ID,n_backbone_matches,SMILES,mol,clean_smiles
0,ABOMOT,10,[H]O[H].[H]c1c([H])c(C([H])([H])[C@@]2([H])C(=...,<rdkit.Chem.rdchem.Mol object at 0x19ffd9230>,C[C@@H]1NC(=O)[C@H](Cc2ccc([N+](=O)[O-])cc2)NC...
1,ABOMOT,10,[H]O[H].[H]c1c([H])c(C([H])([H])[C@@]2([H])C(=...,<rdkit.Chem.rdchem.Mol object at 0x19ffd92a0>,C[C@@H]1NC(=O)[C@H](Cc2ccc([N+](=O)[O-])cc2)NC...
2,ABOMOT,10,[H]O[H].[H]c1c([H])c(C([H])([H])[C@@]2([H])C(=...,<rdkit.Chem.rdchem.Mol object at 0x19ffd8d60>,C[C@@H]1NC(=O)[C@H](Cc2ccc([N+](=O)[O-])cc2)NC...
3,ABOMOT,10,[H]O[H].[H]c1c([H])c(C([H])([H])[C@@]2([H])C(=...,<rdkit.Chem.rdchem.Mol object at 0x19ffd9310>,C[C@@H]1NC(=O)[C@H](Cc2ccc([N+](=O)[O-])cc2)NC...
4,ABOMOT,10,[H]O[H].[H]c1c([H])c(C([H])([H])[C@@]2([H])C(=...,<rdkit.Chem.rdchem.Mol object at 0x19ffd93f0>,C[C@@H]1NC(=O)[C@H](Cc2ccc([N+](=O)[O-])cc2)NC...


In [26]:
def get_macrocycle_ring_sizes(mol):
    """Return sorted list of ring sizes for rings with ≥7 atoms."""
    ring_info = mol.GetRingInfo()
    sizes = sorted([len(r) for r in ring_info.AtomRings() if len(r) >= 7])
    return sizes

def compute_descriptors(smi):
    """Return dict of 2-D and ring descriptors for a SMILES string."""
    empty = {
        'MW': None, 'cLogP': None, 'HBD': None, 'HBA': None,
        'TPSA': None, 'RotBonds': None, 'Fsp3': None,
        'n_rings': None,
    }
    if pd.isna(smi):
        return empty
    mol = Chem.MolFromSmiles(str(smi))
    if mol is None:
        return empty
    try:
        mol.UpdatePropertyCache(strict=False)
        Chem.FastFindRings(mol)
    except Exception:
        return empty

    
    return {
        'MW':               round(Descriptors.ExactMolWt(mol), 3),
        'cLogP':            round(Descriptors.MolLogP(mol), 3),
        'HBD':              rdMolDescriptors.CalcNumHBD(mol),
        'HBA':              rdMolDescriptors.CalcNumHBA(mol),
        'TPSA':             round(Descriptors.TPSA(mol), 3),
        'RotBonds':         rdMolDescriptors.CalcNumRotatableBonds(mol),
        'Fsp3':             round(rdMolDescriptors.CalcFractionCSP3(mol), 4),
        'n_rings':          mol.GetRingInfo().NumRings(),
    }

In [27]:
csd_unique_clean['source']       = 'CSD'
all_db_clean['source']  = 'MacrocycleDB'
ttd_clean['source']      = 'AllDrugs'

all_hits = pd.concat([csd_unique_clean, all_db_clean, ttd_clean],
                      ignore_index=True)
print('Total rows before dedup:', len(all_hits))



all_hits = all_hits.drop_duplicates(subset='clean_smiles', keep='first').reset_index(drop=True)
print('Total unique hits:', len(all_hits))   # expect ~893

Total rows before dedup: 5366
Total unique hits: 727


In [28]:
desc_df = all_hits['clean_smiles'].apply(lambda s: pd.Series(compute_descriptors(s)))
all_hits_desc = pd.concat([all_hits.drop(columns=['clean_smiles']), desc_df], axis=1)

print('Rows with valid MW:', all_hits_desc['MW'].notna().sum())
all_hits_desc.head(3)

Rows with valid MW: 726


,CSD_ID,n_backbone_matches,SMILES,source,mol,ID,Unnamed: 0,row_index,canon_smiles,MW,cLogP,HBD,HBA,TPSA,RotBonds,Fsp3,n_rings
0,ABOMOT,10,[H]O[H].[H]c1c([H])c(C([H])([H])[C@@]2([H])C(=...,CSD,<rdkit.Chem.rdchem.Mol object at 0x1a3bbeff0>,NaN,NaN,NaN,NaN,1218.437,-0.233,8.0,18.0,445.98,12.0,0.3929,7.0
1,ABUDUW,14,O.O.O.O.O.O.O.O.O.O.O.O.O.O.O.[H]N1C(=O)[C@@](...,CSD,<rdkit.Chem.rdchem.Mol object at 0x1a3bbfc30>,NaN,NaN,NaN,NaN,1403.937,2.264,8.0,16.0,369.88,21.0,0.8000,1.0
2,ACTDGU10,8,Cc1ccc(C(=O)N[C@@H]2C(=O)N[C@H](C(C)C)C(=O)N3C...,CSD,<rdkit.Chem.rdchem.Mol object at 0x1a3bbfbc0>,NaN,NaN,NaN,NaN,1256.644,1.674,7.0,18.0,358.37,8.0,0.6129,7.0


## get structutal information from entries with a structure

In [29]:
from ccdc.io import EntryReader
import math

BACKBONE_SMARTS = '[NX3,NX4;R]-[CX4;R]-[CX3;R](=[OX1])'

def csd_backbone_torsions(csd_id: str) -> list[dict]:
    """
    Fetch a CSD entry by identifier and return per-bond backbone torsion
    angles (degrees) for every N-Calpha-C(=O)-N unit in the macrocycle.
    Returns a list of dicts: {csd_id, bond_idx, atom_labels, torsion_deg}
    """
    results = []
    try:
        reader = EntryReader('CSD')
        entry  = reader.entry(csd_id.upper())
        crystal = entry.crystal
        mol = crystal.molecule

        mol_main = max(mol.components, key=lambda m: m.molecular_weight)
        mol_main.assign_bond_types()
        mol_main.add_hydrogens()

        coords = {a.label: a.coordinates for a in mol_main.atoms
                  if a.coordinates is not None}

        mol_main.standardise_aromatic_bonds()
        rdmol = Chem.MolFromSmiles(mol_main.smiles)
        if rdmol is None:
            return results

        query = Chem.MolFromSmarts('[NX3,NX4;R:1]-[CX4;R:2]-[CX3;R:3](=[OX1])-[NX3,NX4;R:4]')
        matches = rdmol.GetSubstructMatches(query)

        atom_labels = [a.label for a in mol_main.atoms]

        def _coord(idx):
            label = atom_labels[idx]
            return coords.get(label)

        def _torsion(p1, p2, p3, p4):
            """Calculate torsion angle in degrees from four 3-D points."""
            import numpy as np
            b1 = np.array(p2) - np.array(p1)
            b2 = np.array(p3) - np.array(p2)
            b3 = np.array(p4) - np.array(p3)
            n1 = np.cross(b1, b2);  n2 = np.cross(b2, b3)
            n1 /= (np.linalg.norm(n1) + 1e-12)
            n2 /= (np.linalg.norm(n2) + 1e-12)
            m1 = np.cross(n1, b2 / (np.linalg.norm(b2) + 1e-12))
            x = np.dot(n1, n2);  y = np.dot(m1, n2)
            return math.degrees(math.atan2(y, x))

        for bond_idx, (i1, i2, i3, i4) in enumerate(matches):
            c1, c2, c3, c4 = _coord(i1), _coord(i2), _coord(i3), _coord(i4)
            if None in (c1, c2, c3, c4):
                continue
            labels = '-'.join([atom_labels[j] for j in (i1, i2, i3, i4)])
            results.append({
                'csd_id':      csd_id,
                'bond_idx':    bond_idx,
                'atom_labels': labels,
                'torsion_deg': round(_torsion(c1, c2, c3, c4), 2),
            })
    except Exception as e:
        print(f'[CSD] {csd_id}: {e}')
    return results


In [30]:
csd_id_col = 'CSD_ID'

csd_rows = all_hits_desc[all_hits_desc['source'] == 'CSD'].copy()
csd_rows = csd_rows[csd_rows[csd_id_col].notna()]

print(f'CSD hits with ID: {len(csd_rows)}')

csd_torsion_records = []
for csd_id in csd_rows[csd_id_col].unique():
    csd_torsion_records.extend(csd_backbone_torsions(csd_id))

csd_torsions_df = pd.DataFrame(csd_torsion_records)
print(f'Torsion entries (CSD): {len(csd_torsions_df)}')
csd_torsions_df.head()

CSD hits with ID: 488
[CSD] ABOMOT: CSD Data is not available in this installation. Cannot load CSD data from None
[CSD] ABUDUW: CSD Data is not available in this installation. Cannot load CSD data from None
[CSD] ACTDGU10: CSD Data is not available in this installation. Cannot load CSD data from None
[CSD] ACUMOC: CSD Data is not available in this installation. Cannot load CSD data from None
[CSD] AENNIC10: CSD Data is not available in this installation. Cannot load CSD data from None
[CSD] AHELEG: CSD Data is not available in this installation. Cannot load CSD data from None
[CSD] AJAGEX: CSD Data is not available in this installation. Cannot load CSD data from None
[CSD] ALASAR: CSD Data is not available in this installation. Cannot load CSD data from None
[CSD] ALAZOD: CSD Data is not available in this installation. Cannot load CSD data from None
[CSD] ALPRAL10: CSD Data is not available in this installation. Cannot load CSD data from None
[CSD] AMIFAE: CSD Data is not available in

/Users/danakatz/miniforge3/envs/csdapi/lib/python3.11/site-packages/ccdc/io.py:121: UserWarning: Cannot locate the CSD database.
For further help with installing and configuring data please visit
the support page at https://www.ccdc.cam.ac.uk/csds_install_help
  warnings.warn(_CSDDatabaseLocator.get_location_warning())
/Users/danakatz/miniforge3/envs/csdapi/lib/python3.11/site-packages/ccdc/io.py:154: UserWarning: Cannot locate the CSD database.
For further help with installing and configuring data please visit
the support page at https://www.ccdc.cam.ac.uk/csds_install_help
  warnings.warn(_CSDDatabaseLocator.get_location_warning())

""


ile format has been registered in your program's main()